# From `System` to compiled dynamics primitives

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/07_compile.ipynb)

Minilink models are **textbook `System` objects** — you write `f(x, u, t, params)` and
output maps once. **Compilation** lowers a leaf or wired diagram into a flat
**evaluator**: a few fast dynamic primitives (`f`, `outputs`, `step`,
`rollout`) backed by NumPy or JAX.

This notebook is a focused introduction to that pipeline:

1. [The model stays textbook](#1.-The-model-stays-textbook)
2. [`compile(backend)` → evaluator](#2.-compilebackend--evaluator)
3. [What compilation does to a diagram](#3.-What-compilation-does-to-a-diagram)
4. [The step model: dynamics primitives](#4.-The-step-model-dynamics-primitives)
5. [NumPy vs JAX backends](#5.-NumPy-vs-JAX-backends)
6. [Compiled methods inside JAX transformations (brief)](#6.-Compiled-methods-inside-JAX-transformations-(brief))
7. [Where evaluators plug in](#7.-Where-evaluators-plug-in)

**Run locally** with minilink installed (`pip install -e .` from the repo root), or
open on Colab — the setup cell detects Colab and clones the repo automatically.

**Next:** [`08_optimization`](08_optimization.ipynb)


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. The model stays textbook

A continuous-time plant implements one dynamics equation plus boundary outputs:

    dx = f(x, u, t, params)
    y  = h(x, u, t, params)   # one map per output port

Simulation facades (`compute_trajectory`, …) call **compile internally**, but you
can also compile once and hold the evaluator when you need repeated fast calls
(MPC inner loops, benchmarks, custom integrators, NLP defects).


In [ ]:
import numpy as np

from minilink import Pendulum

plant = Pendulum()
plant.params["d"] = 0.2
plant.x0 = np.array([0.5, 0.0])

x = plant.x0.copy()
u = np.array([0.0])

print("plant.f(x, u) =", plant.f(x, u))
plant

## 2. `compile(backend)` → evaluator

`sys.compile(backend="numpy" | "jax")` returns a **typed evaluator**:

| Model kind | Evaluator | Primary primitives |
| --- | --- | --- |
| `DynamicSystem` leaf or `DiagramSystem` | `DynamicEvaluator` / `DiagramEvaluator` | `.f`, `.outputs`, the step model `.step` / `.rollout` |
| `StepSystem` / step diagram | `StepEvaluator` / `StepDiagramEvaluator` | `.step`, state-only `.rollout`, `.outputs` |
| static leaf (`n=0`) | `StaticEvaluator` | `.outputs` only |

Every method is **one body** taking `params=None` (`None` reads the parameters compiled
in): eager on NumPy, and on JAX compiled once with `jax.jit`, so it is fast from Python
and still composes inside an outer `jit` / `grad` / `vmap` (Section 6).


In [ ]:
np_eval = plant.compile(backend="numpy")
jax_eval = plant.compile(backend="jax")

print(type(np_eval).__name__, "backend =", np_eval.backend)
print(type(jax_eval).__name__, "backend =", jax_eval.backend)
print("state dim n =", jax_eval.n, ", input dim m =", jax_eval.m)

dx_np = np_eval.f(x, u)
dx_jax = np.asarray(jax_eval.f(x, u))
print("np_eval.f  =", dx_np)
print("jax_eval.f =", dx_jax)

## 3. What compilation does to a diagram

On a **wired diagram**, compilation is a two-stage lowering:

1. **Topology pass** — resolve port dependencies, detect algebraic loops, produce
   a fixed evaluation order.
2. **`ExecutionPlan`** — flatten subsystem `f` / port computes into one state
   vector and one internal signal buffer; wire gathers with `gather_u`.

The evaluator then exposes the **same primitive API** as a leaf: one call to `.f`
runs the whole diagram without recursive Python dispatch.


In [ ]:
from minilink import Impedance, Step

step = Step()
step.params["final_value"] = np.array([1.0])
step.params["step_time"] = 2.0

ctl = Impedance(Kp=50.0, Kd=5.0)

diagram = step >> ctl @ plant
diagram

In [ ]:
diag_eval = diagram.compile(backend="jax", verbose=True)

print("plan state_dim:", diag_eval.plan.state_dim)
print("plan signal_dim:", diag_eval.plan.signal_dim)
print("port ops:", len(diag_eval.plan.port_ops))
print("state ops:", len(diag_eval.plan.state_ops))

x_diag = np.zeros(diagram.n)
u_diag = diagram.get_u_from_input_ports()
print("diagram.f(x, u) =", np.asarray(diag_eval.f(x_diag, u_diag)))

## 4. The step model: dynamics primitives

A compiled **dynamics evaluator** is a small toolbox of numerical kernels:

| Layer | Examples | Role |
| --- | --- | --- |
| **Dynamics** | `f(x, u, t, params)` | One evaluation of $\dot x$ |
| **Outputs** | `outputs(x, u, t, params)` | Boundary port dict (not diagram internals) |
| **One step** | `step(x, u, t, params, *, dt, solver, substeps)` | The plant one period $\Delta t$ later, `u` held |
| **Rollout** | `rollout(x0, u_seq, t0, dt, params, *, solver, substeps, hold)` | Many steps in one call |

`solver` is `"rk4"` (default) or `"euler"`; `substeps` splits each period; `hold="zoh"`
holds each row of `u_seq` over its period, `hold="foh"` reads `u_seq` as knots, linear
between them. `params=None` reads the compiled parameters.

In [ ]:
dt = 0.01
t0 = 0.0

# --- dynamics + outputs ---
dx = jax_eval.f(x, u, t0)
y = jax_eval.outputs(x, u, t0)
print("f(x,u)     =", np.asarray(dx))
print("outputs    =", {k: np.asarray(v) for k, v in y.items()})

# --- one RK4 step, u held ---
x1 = np.asarray(jax_eval.step(x, u, t0, dt=dt))
print("step       =", x1)

# --- rollout: hold u over a uniform grid ---
n_steps = 50
u_seq = np.tile(u, (n_steps, 1))
x_path = np.asarray(jax_eval.rollout(x, u_seq, t0, dt))
print("rollout shape:", x_path.shape, "  final x =", x_path[-1])

The same step with finer integration inside one hold window is `substeps`:

Discrete **step** evaluators expose `.step` and a state-only `.rollout(k, x, u)`.
A sampled loop (`(law % dt) @ plant`) is itself a step system: its evaluator's
`rollout_dense` keeps the plant's substeps, and `compute_trajectory` returns them.

In [ ]:
x_held = jax_eval.step(x, u, t0, dt=n_steps * dt, substeps=n_steps)
print("one hold window, 50 substeps:", np.asarray(x_held), "≈", x_path[-1])

## 5. NumPy vs JAX backends

Both backends expose the **same public API**. The difference is execution:

- **NumPy** — eager Python loops for rollouts; good for debugging and small models.
- **JAX** — `.f` and rollouts JIT to XLA; first call pays compile cost, then fast.

On a diagram, the win is eliminating recursive port dispatch — plus JAX fusion
when the backend is `"jax"`.


In [ ]:
import time

n = 2000
xc = np.zeros(plant.n)
uc = np.zeros(plant.m)

for label, fn in [
    ("plant.f (recursive)", lambda: plant.f(xc, uc)),
    ("NumPy evaluator.f", lambda: np_eval.f(xc, uc)),
    ("JAX evaluator.f", lambda: jax_eval.f(xc, uc)),
]:
    out = fn()
    if hasattr(out, "block_until_ready"):
        out.block_until_ready()
    t0_b = time.perf_counter()
    for _ in range(n):
        out = fn()
        if hasattr(out, "block_until_ready"):
            out.block_until_ready()
    dt_b = time.perf_counter() - t0_b
    print(f"{label:22s}: {1e6 * dt_b / n:6.1f} µs/call")

## 6. Compiled methods inside JAX transformations (brief)

On JAX each method is already JIT-compiled, and a compiled function composes: an *outer*
`jax.jit(jax.grad(...))` or `jax.vmap` traces straight through `.f`, `.step` and
`.rollout`, with the same numbers as the uncompiled body. Pass `params` to
differentiate with respect to them. Use the JAX backend when you need AD through compiled
dynamics.



In [ ]:
import jax
import jax.numpy as jnp

x_j = jnp.array(x)
u_j = jnp.array(u)
params = {"m": 1.0, "l": 1.0, "I": 1.0, "gravity": 9.81, "d": 0.2}

A = jax.jacfwd(lambda xx: jax_eval.f(xx, u_j, 0.0))(x_j)
print("A = df/dx:\n", np.asarray(A).round(3))

# The same derivative as a callable of (x, u, t, params) — the function form
# behind plant.jacobian("f", "x"); wrap it in jax.jit for hot loops.
dfdx = jax_eval.jacobian("f", "x")
print("evaluator.jacobian matches:", np.allclose(np.asarray(dfdx(x_j, u_j, 0.0, params)), np.asarray(A)))

d_d = jax.grad(
    lambda d_val: jnp.sum(
        jax_eval.f(
            x_j, u_j, 0.0, {"m": 1.0, "l": 1.0, "I": 1.0, "gravity": 9.81, "d": d_val}
        )
        ** 2
    )
)(jnp.array(0.2))
print("d(loss)/d(damping) =", float(d_d))

## 7. Where evaluators plug in

Compiled evaluators are the **fast math layer** behind higher-level workflows:

| Consumer | Typical evaluator use |
| --- | --- |
| `Simulator` | `compile(backend=...)` then integrate (`rk4`, `euler`, `scipy_ivp`, …) |
| `compute_trajectory` façade | compile + solve; caches `Trajectory` |
| Trajectory optimization / MPC | JAX `.f`, `.step` and `.rollout` for defects and gradients |
| Sampled loop `(law % dt) @ plant` | one compiled plan per base tick: the clocked blocks fire, the plant's step model integrates the period |
| Custom scripts | hold `evaluator = sys.compile(...)` and call primitives directly |

**Mental model:** write equations on `System` objects → **compile once** → call
primitives many times. Facades and simulators are thin orchestration on top.

